In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pickle as pkl
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "speeches.parquet").is_file()
)
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

input_path = PROJECT_ROOT / "data" / "data_with_topics.pkl"
if input_path.exists():
    df = pd.read_pickle(input_path)
else:
    df = pd.read_parquet(PROJECT_ROOT / "data" / "speeches.parquet")

topic_path = PROJECT_ROOT / "data" / "policy_topics.parquet"
if "topic_label" not in df.columns and topic_path.exists():
    topics = pd.read_parquet(topic_path)
    df = df.merge(topics[["original_row_idx", "topic_label"]],
                  on="original_row_idx", how="left", validate="one_to_one")

df['moral_rating'] = pd.to_numeric(df['moral_rating'], errors='coerce')
df = df.dropna(subset=['moral_rating'])
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year

PARTY_SV_NORMALIZE = {
    "Moderata samlingspartiet": "Moderaterna",
    "Folkpartiet liberalerna": "Liberalerna",
    "Folkpartiet": "Liberalerna",
    "Kristdemokratiska Samhällspartiet": "Kristdemokraterna",
    "Vänsterpartiet Kommunisterna": "Vänsterpartiet",
}

REMOVE_PARTIES = {"Utan partibeteckning", "no party"}
df.head()

In [ ]:
# only include from 1991 election onward
df = df[df['moral_rating'] != 0]
df =df[df['date'] >= pd.to_datetime("1991-09-15")]

In [ ]:
stats = {}

In [ ]:
stats['number of speeches'] = len(df)
df['party_name_norm'] = df['party_name'].replace(PARTY_SV_NORMALIZE)
print(len(df))
df = df[~df['party_name_norm'].isin(REMOVE_PARTIES)]
stats['number of valid speeches'] = len(df)
TOPICS_AVAILABLE = "topic_label" in df.columns and df["topic_label"].notna().all()
if not TOPICS_AVAILABLE:
    print("Policy-topic labels unavailable: skipping topic-controlled models and figures.")


In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]

In [ ]:

government_periods = {
    (pd.to_datetime("1991-10-04"), pd.to_datetime("1994-10-06")): [
        "Centerpartiet", "Liberalerna", "Moderaterna", "Kristdemokraterna"
    ],
    (pd.to_datetime("1994-10-07"), pd.to_datetime("1996-03-21")): [
        "Socialdemokraterna"
    ],
    (pd.to_datetime("1996-03-22"), pd.to_datetime("1998-09-19")): [
        "Socialdemokraterna"
    ],
    (pd.to_datetime("1998-09-20"), pd.to_datetime("2002-09-14")): [
        "Socialdemokraterna"
    ],
    (pd.to_datetime("2002-09-15"), pd.to_datetime("2006-10-02")): [
        "Socialdemokraterna"
    ],
    (pd.to_datetime("2006-10-03"), pd.to_datetime("2010-09-18")): [
        "Centerpartiet", "Liberalerna", "Moderaterna", "Kristdemokraterna"
    ],
    (pd.to_datetime("2010-09-19"), pd.to_datetime("2014-10-02")): [
        "Centerpartiet", "Liberalerna", "Moderaterna", "Kristdemokraterna"
    ],
    (pd.to_datetime("2014-10-03"), pd.to_datetime("2019-01-20")): [
        "Socialdemokraterna", "Miljöpartiet"
    ],
    (pd.to_datetime("2019-01-21"), pd.to_datetime("2021-11-29")): [
        "Socialdemokraterna", "Miljöpartiet"
    ],
    (pd.to_datetime("2021-11-30"), pd.to_datetime("2022-10-17")): [
        "Socialdemokraterna"
    ],
    (pd.to_datetime("2022-10-18"), pd.to_datetime("2026-01-01")): [
        "Liberalerna", "Kristdemokraterna", "Moderaterna"
    ],
}

support_periods = {
    (pd.to_datetime("1991-10-04"), pd.to_datetime("1994-10-06")): [],
    (pd.to_datetime("1994-10-07"), pd.to_datetime("1996-03-21")): [
        "Vänsterpartiet", "Centerpartiet"
    ],
    (pd.to_datetime("1996-03-22"), pd.to_datetime("1998-09-19")): [
        "Centerpartiet"
    ],
    (pd.to_datetime("1998-09-20"), pd.to_datetime("2002-09-14")): [
        "Vänsterpartiet", "Miljöpartiet"
    ],
    (pd.to_datetime("2002-09-15"), pd.to_datetime("2006-10-02")): [
        "Vänsterpartiet", "Miljöpartiet"
    ],
    (pd.to_datetime("2006-10-03"), pd.to_datetime("2010-09-18")): [],
    (pd.to_datetime("2010-09-19"), pd.to_datetime("2014-10-02")) : [],
    (pd.to_datetime("2014-10-03"), pd.to_datetime("2019-01-20")): [
        "Vänsterpartiet"
    ],
    (pd.to_datetime("2019-01-21"), pd.to_datetime("2021-11-29")): [
        "Vänsterpartiet", "Centerpartiet", "Liberalerna"
    ],
    (pd.to_datetime("2021-11-30"), pd.to_datetime("2022-10-17")): [],
    (pd.to_datetime("2022-10-18"), pd.to_datetime("2026-01-01")): [
        "Sverigedemokraterna"
    ],
}


In [ ]:
def get_power_status_extended(row):
    for (start_date, end_date), gov_parties in government_periods.items():
        if start_date <= row['date'] < end_date:
            if row['party_name_norm'] in gov_parties:
                return "in power"
            support_parties = support_periods.get((start_date, end_date), [])
            if row['party_name_norm'] in support_parties:
                return "supporting"
            return "not in power"
    return "not in power"


In [ ]:
df['power_status'] = df[['date', 'party_name_norm']].apply(get_power_status_extended, axis=1)

In [ ]:
if TOPICS_AVAILABLE:
    # Topic distribution
    topic_dist = (
        df["topic_label"]
        .value_counts(dropna=False)
        .rename_axis("Topic")
        .reset_index(name="Count")
    )

    topic_dist["Percentage"] = (
        100 * topic_dist["Count"] / topic_dist["Count"].sum()
    ).round(1)

    topic_dist

In [ ]:
if TOPICS_AVAILABLE:
    latex_table = topic_dist.to_latex(
        index=False,
        caption="Topic distribution of parliamentary speeches",
        label="tab:topic_distribution",
        column_format="lrr",
        escape=True,
    )

    print(latex_table)
    topic_dist.to_csv(TABLE_DIR / "table_a5_1.csv", index=False)
    (TABLE_DIR / "table_a5_1.tex").write_text(latex_table, encoding="utf-8")


In [ ]:
#plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman'], 'axes.linewidth': 1.5})

PARTY_EN = {"Vänsterpartiet": "Left Party", "Miljöpartiet": "Green Party", "Socialdemokraterna": "Social Democrats",
            "Centerpartiet": "Center Party", "Liberalerna": "Liberals", "Kristdemokraterna": "Christian Democrats",
            "Moderaterna": "Moderates", "Sverigedemokraterna": "Sweden Democrats", "Ny demokrati": "New Democracy"}
PARTY_ORDER = ["Left Party", "Green Party", "Social Democrats", "Center Party", "Liberals", "Christian Democrats", "Moderates", "Sweden Democrats"]
PARTY_COLORS = {"Left Party": '#A61C2B', "Green Party": '#3A7734', "Social Democrats": '#C64256', "Center Party": '#6A8B3F',
                "Liberals": '#3B6FB6', "Christian Democrats": '#354E9C', "Moderates": '#2A81B8', "Sweden Democrats": '#D97706', "New Democracy": '#ffff00'}

df_viz = df.copy()
df_viz['party_en'] = df_viz['party_name_norm'].map(PARTY_EN).fillna(df_viz['party_name_norm'])
party_means = df_viz.groupby(['year', 'party_en'])['moral_rating'].mean().unstack()
party_order = PARTY_ORDER + [p for p in party_means.columns if p not in PARTY_ORDER]
party_means = party_means.reindex(columns=party_order)
mean_ratings = df_viz.groupby('year')['moral_rating'].mean()
std_error = df_viz.groupby('year')['moral_rating'].std() / np.sqrt(df_viz.groupby('year').size())

fig, ax = plt.subplots(figsize=(16, 8), dpi=300)
ax.set_facecolor('#FEFEFE')
ax.grid(axis='y', linestyle='-', alpha=0.15, linewidth=0.8, color='#666')

for party in party_means.columns:
    lw = 2.8 if party in ["Social Democrats", "Moderates"] else 2.2
    ax.plot(party_means.index, party_means[party], lw=lw, color=PARTY_COLORS.get(party, '#999'), alpha=0.85, label=party)

ax.errorbar(
    mean_ratings.index,
    mean_ratings.values,
    yerr=std_error,
    fmt='o-',
    color='#2B2B2B',
    ecolor='#6E6E6E',
    capsize=4,
    lw=2.8,
    markersize=6,
    capthick=1.4,
    markerfacecolor='white',
    markeredgewidth=1.8,
    label='Overall Average',
    zorder=10
)

ax.set_xlabel('Year', fontsize=15)
ax.set_ylabel('Average Moral Rating', fontsize=15)
#ax.set_title("Average moral rating by year and party", pad=20, fontsize=16)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
ax.legend(title='Political Parties', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11, frameon=True)
plt.tight_layout()
plt.subplots_adjust(right=0.75)
out_path = FIG_DIR / "parties.jpg"
plt.savefig(out_path)
plt.show()


party_means.to_csv(TABLE_DIR / "annual_party_means.csv")


In [ ]:
ELECTIONS = {
    1991: "1991-09-15",
    1994: "1994-09-18",
    1998: "1998-09-20",
    2002: "2002-09-15",
    2006: "2006-09-17",
    2010: "2010-09-19",
    2014: "2014-09-14",
    2018: "2018-09-09",
    2022: "2022-09-11",
    2026: "2026-09-13"
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.transforms as mtransforms


df_plot = df.copy()

# Dates
df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")

# Exclude summer recess months (July–September)
df_plot = df_plot[~df_plot["date"].dt.month.isin([7, 8, 9])]

df_plot = df_plot[df_plot["power_status"].notna()]
df_plot["moral_rating"] = pd.to_numeric(df_plot["moral_rating"], errors="coerce")
df_plot = df_plot.dropna(subset=["date", "moral_rating"])

df_plot["year_month"] = df_plot["date"].dt.to_period("M")
df_plot["power_status"] = np.where(
    df_plot["power_status"] == "supporting",
    "not in power",
    df_plot["power_status"]
)

summary = (
    df_plot
    .groupby(["year_month", "power_status"])["moral_rating"]
    .agg(mean="mean", std="std", n="count")
    .reset_index()
    .sort_values("year_month")
)

summary["se"] = summary["std"] / np.sqrt(summary["n"])
summary["month_ts"] = summary["year_month"].dt.to_timestamp()
summary["month_idx"] = (
    summary["year_month"].dt.year * 12 + summary["year_month"].dt.month
)

election_month_idx = (
    pd.to_datetime(list(ELECTIONS.values()))
    .year * 12
    + pd.to_datetime(list(ELECTIONS.values())).month
)

ELECTION_WINDOW = 2  # months

summary["election_break"] = summary["month_idx"].apply(
    lambda m: any(abs(em - m )< ELECTION_WINDOW for em in election_month_idx)
)


SMOOTH_WINDOW = 5  # months

summary = (
    summary
    .sort_values("month_idx")
    .groupby("power_status", group_keys=False)
    .apply(
        lambda d: d.assign(
            mean_smooth=d["mean"].rolling(
                SMOOTH_WINDOW, center=True, min_periods=1
            ).mean(),
            se_smooth=d["se"].rolling(
                SMOOTH_WINDOW, center=True, min_periods=1
            ).mean()
        )
    )
)


status_colors = {
    "in power": "#1b9e77",
    "not in power": "#7570b3",
    "supporting": "#d95f02"
}


fig, ax = plt.subplots(figsize=(12, 6))

for status, data in summary.groupby("power_status"):
    data = data.sort_values("month_idx")
    color = status_colors.get(status, "black")

    gaps = (
    data["month_idx"].diff().gt(6)
    )
    #data["election_break"] |
    #data["election_break"].shift(fill_value=False)
    #)
    
    segments = gaps.cumsum()


    for _, seg in data.groupby(segments):
        # Uncertainty band
        ax.fill_between(
            seg["month_ts"],
            seg["mean_smooth"] - seg["se_smooth"],
            seg["mean_smooth"] + seg["se_smooth"],
            color=color,
            alpha=0.18,
            linewidth=0
        )

        # Smoothed line
        ax.plot(
            seg["month_ts"],
            seg["mean_smooth"],
            lw=2.2,
            color=color
        )

# -------------------------
# Elections
# -------------------------



ELECTIONS_ = {
    1991: "1991-09-15",
    1994: "1994-09-18",
    1998: "1998-09-20",
    2002: "2002-09-15",
    2006: "2006-09-17",
    2010: "2010-09-19",
    2014: "2014-09-14",
    2018: "2018-09-09",
    2022: "2022-09-11",
    #2026: "2026-09-13"
}
trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

for year, date_str in ELECTIONS_.items():
    election_date = pd.to_datetime(date_str)

    ax.axvline(
        election_date,
        linestyle="-",
        lw=1,
        color="0.1",
        alpha=1,
        zorder=3
    )

    ax.text(
        election_date,
        1.02,
        str(year),
        transform=trans,
        ha="center",
        va="bottom",
        fontsize=9,
        color="0.4"
    )


years = summary["month_ts"].dt.year.unique()

for year in years:
    ax.axvspan(
        pd.to_datetime(f"{year}-07-01"),
        pd.to_datetime(f"{year}-09-30"),
        color="0.9",
        alpha=0.6,
        zorder=0
    )


legend_elements = [
    Line2D([0], [0], color=color, lw=2, label=status)
    for status, color in status_colors.items()
    if status in summary["power_status"].unique()
]

legend_elements.append(
    Line2D([0], [0], color="0.2", lw=8, alpha=0.2, label="summer recess")
)

legend_elements.append(
    Line2D([0], [0], color="0.1", lw=1, linestyle="-", label="election")
)

ax.legend(handles=legend_elements, title="Government status")

#ax.set_title("Average moral rating by month and government status", pad=20)
ax.set_xlabel("Year")
ax.set_ylabel("Average Moral Rating")
#ax.set_xlim(1991, 2023)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
out_path = FIG_DIR / "rolling.jpg"
plt.savefig(out_path)
plt.show()


summary.to_csv(TABLE_DIR / "monthly_government_means.csv", index=False)


# Descriptives

In [ ]:
for party, n in df.groupby(['party_name_norm']).size().items():
    stats[PARTY_EN[party]] = n

In [ ]:
print(stats)

In [ ]:
summary = (
    df.groupby('party_name_norm')['moral_rating']
    .agg(
        N='count',
        Mean='mean',
        SD='std',
        Variance='var',
        Min='min',
        Max='max'
    )
    .round(2)   # nice formatting
)
summary.index = summary.index.map(PARTY_EN)

overall = pd.DataFrame({
    'N': [df['moral_rating'].count()],
    'Mean': [df['moral_rating'].mean()],
    'SD': [df['moral_rating'].std()],
    'Variance': [df['moral_rating'].var()],
    'Min': [df['moral_rating'].min()],
    'Max': [df['moral_rating'].max()]
}, index=['All Parties']).round(2)


summary = summary.sort_values('Mean', ascending = False)

# Append overall row
summary = pd.concat([summary, overall])


summary

In [ ]:
latex_table = summary.to_latex(
    caption="Descriptive statistics of moral rating by party",
    label="tab:moral_by_party",
    float_format="%.2f",
    bold_rows=True
)

print(latex_table)

In [ ]:
def days_to_next_election(date):
    if pd.isna(date):
        return np.nan
    
    future_deltas = (election_dates - date).days
    future_deltas = future_deltas[future_deltas >= 0]

    return future_deltas.min() if len(future_deltas) > 0 else np.nan


In [ ]:
election_dates = pd.to_datetime(list(ELECTIONS.values()))

df["days_to_next_election"] = df["date"].apply(days_to_next_election)

# log makes sense since it's likely not a linear effect, adding one to account for election day as log(1)
df["log_days_to_election"] = np.log(df["days_to_next_election"] + 1)

In [ ]:
anti_establishment = {"Vänsterpartiet" : 1, "Miljöpartiet" : 1, "Socialdemokraterna" : 0, "Centerpartiet": 0, "Liberalerna" : 0, "Kristdemokraterna" :0, "Moderaterna" : 0, "Sverigedemokraterna" :1, "Ny demokrati" : 1}
df['anti_establishment'] = df['party_name_norm'].map(anti_establishment)
print(df['anti_establishment'].unique())

In [ ]:
def get_gov_status_dummy(row):
    if row['power_status'] == 'in power':
        return 1
    else:
        return 0

df['gov_status'] = df.apply(get_gov_status_dummy, axis = 1)
print(df['gov_status'].unique())

In [ ]:
df_reg = df.copy()
df_reg['year_month'] = df_reg['date'].dt.to_period('M')

In [ ]:
df_reg_ = (
    df_reg[['person_id', 'year_month', 'party_name_norm', 'moral_rating', 'gov_status', 'anti_establishment']]
    .groupby(['person_id', 'year_month'], as_index=False)
        .agg(
            moral_rating=('moral_rating', 'mean'),
            party=('party_name_norm', 'first'),
            anti_establishment=('anti_establishment', 'first'),
            gov_status=('gov_status', 'first')
            )
)

In [ ]:
election_periods = election_dates.to_period('M')

In [ ]:
def months_to_next_election(date):
    if pd.isna(date):
        return np.nan

    date = pd.Period(date, freq='M')
    future = election_periods[election_periods >= date]

    if len(future) == 0:
        return np.nan

    next_election = future.min()

    return (
        (next_election.year - date.year) * 12
        + (next_election.month - date.month)
    )


In [ ]:
df_reg_["months_to_next_election"] = df_reg_["year_month"].apply(months_to_next_election)


In [ ]:
# Log months to election instead of days when we aggregate
df_reg_["log_months_to_election"] = np.log(df_reg_["months_to_next_election"] + 1)

# Regression

In [ ]:
import statsmodels.api as sm
import numpy as np

df_reg = df_reg_.dropna(
    subset=[
        "moral_rating",
        "months_to_next_election",
        "gov_status",
        "anti_establishment"
    ]
)


In [ ]:
# find election terms
df_reg['election_term'] = (
    election_periods
        .searchsorted(df_reg['year_month'], side='right')
        - 1
)

# Speech-level regression

In [ ]:
df_speech = df[[
    "moral_rating",
    "gov_status",
    "days_to_next_election",
    "person_id",
    "party_name_norm",
    "date",
    ] + (["topic_label"] if TOPICS_AVAILABLE else [])].copy()

df_speech = df_speech.rename(columns = {'party_name_norm' : 'party'})
df_speech['election_term'] = (
    election_periods
        .searchsorted(pd.PeriodIndex(df_speech["date"], freq="M"), side='right')
        - 1
)

df_speech['covid_year'] = df_speech["date"].dt.year.isin([2020]).astype(int)
df_speech['refugee_crisis'] = df_speech["date"].dt.year.isin([2015]).astype(int)


df_speech.head()

In [ ]:
if TOPICS_AVAILABLE:
    # topic occurances
    topic_summary = (
        df_speech["topic_label"]
        .value_counts(dropna=False)
        .rename_axis("topic_label")
        .reset_index(name="count")
    )

    topic_summary["percentage"] = (
        topic_summary["count"] / len(df) * 100
    ).round(2)

    topic_summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_plot = df_speech.reset_index().copy()

# Ensure numeric
df_plot["moral_rating"] = pd.to_numeric(df_plot["moral_rating"], errors="coerce")
df_plot["days_to_next_election"] = pd.to_numeric(
    df_plot["days_to_next_election"],
    errors="coerce"
)

# Keep only observations up to 2 years before election
df_plot = df_plot[
    (df_plot["days_to_next_election"] >= 0) &
    (df_plot["days_to_next_election"] <= 365)
].copy()

df_plot = df_plot.dropna(subset=["moral_rating", "days_to_next_election"])

# Equal-length bins: 90-day intervals up to 730 days
bins = list(range(0, 345, 20)) + [365]

labels = [
    f"{bins[i]}-{bins[i+1] - 1}"
    for i in range(len(bins) - 1)
]

df_plot["election_bin_equal_length"] = pd.cut(
    df_plot["days_to_next_election"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# Summarize by bin
bin_summary = (
    df_plot
    .groupby("election_bin_equal_length", observed=True)
    .agg(
        mean_moral_rating=("moral_rating", "mean"),
        n=("moral_rating", "size"),
        mean_days=("days_to_next_election", "mean")
    )
    .reset_index()
)

# Add n to labels
bin_summary["label"] = (
    bin_summary["election_bin_equal_length"].astype(str)
)

# Plot
plt.figure(figsize=(9, 5))

plt.plot(
    bin_summary["label"],
    bin_summary["mean_moral_rating"],
    marker="o",
    linewidth=2
)

plt.xlabel("Days to next election")
plt.ylabel("Mean moral rating")
#plt.title("Moral rating by election proximity, up to two years before election")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    FIG_DIR / "equal_length_bins_two_years_before_election.jpg",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
bin_summary.to_csv(TABLE_DIR / "election_bins.csv", index=False)


In [ ]:
df_speech = df_speech.set_index(["person_id", "date"])

df_speech["months_to_election"] = df_speech["days_to_next_election"] / 30.44

df_speech["log_months_to_election"] = np.log(df_speech["months_to_election"] + 1)

In [ ]:
from linearmodels.panel import PanelOLS
from stargazer.stargazer import Stargazer

# without gov status and with party * election interactions

mod1 = PanelOLS.from_formula(
    "moral_rating ~ gov_status + log_months_to_election + EntityEffects + C(election_term)",
    data=df_speech
).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)


if TOPICS_AVAILABLE:
    mod2 = PanelOLS.from_formula(
        "moral_rating ~ gov_status + log_months_to_election + EntityEffects + C(topic_label) + C(election_term)",
        data=df_speech
    ).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

    mod3 = PanelOLS.from_formula(
        "moral_rating ~ gov_status*log_months_to_election + EntityEffects + C(topic_label) + C(election_term)",
        data=df_speech
    ).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
    mp_models = [mod1, mod2]
    mp_interaction = mod3
else:
    mp_models = [mod1]

pd.DataFrame({"estimate": mod1.params, "se": mod1.std_errors}).to_csv(TABLE_DIR / "mp_fixed_effects.csv")


In [ ]:
if TOPICS_AVAILABLE:
    print(mod2.rsquared_overall)

In [ ]:
if TOPICS_AVAILABLE:
    # with event controls

    mod4 = PanelOLS.from_formula(
        "moral_rating ~ gov_status*log_months_to_election + EntityEffects + C(topic_label) + C(election_term) + covid_year + refugee_crisis",
        data=df_speech
    ).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
    mod4.summary

In [ ]:
if TOPICS_AVAILABLE:
    stargazer = Stargazer([mod4])

    latex_table = stargazer.render_latex()

    print(latex_table)
    (TABLE_DIR / "table_a6_2.tex").write_text(latex_table, encoding="utf-8")

In [ ]:
import statsmodels.formula.api as smf

df_speech_no_index = df_speech.reset_index()

person_codes = df_speech_no_index["person_id"].astype("category").cat.codes
day_codes = df_speech_no_index["date"].astype("category").cat.codes
speech_groups = np.column_stack((person_codes, day_codes))
print(speech_groups.shape)
print(len(df_speech_no_index))


mod1 = smf.ols(
    "moral_rating ~  gov_status +  C(party, Treatment(reference='Socialdemokraterna')) + log_months_to_election + C(election_term)",
    data=df_speech_no_index
).fit(cov_type="cluster",
    cov_kwds={"groups":speech_groups})


if TOPICS_AVAILABLE:
    mod2 = smf.ols(
        "moral_rating ~  gov_status +  C(party, Treatment(reference='Socialdemokraterna')) + log_months_to_election +  C(topic_label)  + C(election_term)",
        data=df_speech_no_index
    ).fit(cov_type="cluster",
        cov_kwds={"groups":speech_groups})

    mod3 = smf.ols(
        "moral_rating ~  gov_status *log_months_to_election + C(party, Treatment(reference='Socialdemokraterna')) +  C(topic_label) + C(election_term)",
        data=df_speech_no_index
    ).fit(cov_type="cluster",
        cov_kwds={"groups":speech_groups})
    stargazer = Stargazer([mp_interaction, mod3])
    latex_table = stargazer.render_latex()
    (TABLE_DIR / "table_a6_4.tex").write_text(latex_table, encoding="utf-8")
    print(latex_table)
    table_2_models = [mod1, mod2, *mp_models]
else:
    table_2_models = [mod1, *mp_models]

stargazer = Stargazer(table_2_models)
latex_table = stargazer.render_latex()
print(latex_table)
(TABLE_DIR / ("table_2.tex" if TOPICS_AVAILABLE else "table_2_partial.tex")).write_text(latex_table, encoding="utf-8")
pd.DataFrame({"estimate": mod1.params, "se": mod1.bse}).to_csv(TABLE_DIR / "party_control_coefficients.csv")


In [ ]:
if TOPICS_AVAILABLE:
    mod4 = smf.ols(
        "moral_rating ~  log_months_to_election + C(party, Treatment(reference='Socialdemokraterna')) +  C(topic_label) + C(election_term)",
        data=df_speech_no_index
    ).fit(cov_type="cluster",
        cov_kwds={"groups":speech_groups})


    mod5 = smf.ols(
        "moral_rating ~  gov_status + log_months_to_election + C(party, Treatment(reference='Socialdemokraterna')) +  C(topic_label) + C(election_term)",
        data=df_speech_no_index
    ).fit(cov_type="cluster",
        cov_kwds={"groups":speech_groups})

    mod6 = smf.ols(
        "moral_rating ~  gov_status  + log_months_to_election * C(party, Treatment(reference='Socialdemokraterna')) +  C(topic_label) + C(election_term)",
        data=df_speech_no_index
    ).fit(cov_type="cluster",
        cov_kwds={"groups":speech_groups})



    #print(mod4.summary())

    stargazer = Stargazer([mod4, mod5, mod6])

    latex_table = stargazer.render_latex()



    print(latex_table)
    (TABLE_DIR / "table_a6_3.tex").write_text(latex_table, encoding="utf-8")

In [ ]:
if TOPICS_AVAILABLE:
    import pandas as pd
    import matplotlib.pyplot as plt
    import statsmodels.formula.api as smf

    # Reference values
    gov_ref = df_speech_no_index["gov_status"].mean()
    election_term_ref = df_speech_no_index["election_term"].mode()[0]
    log_months_ref = df_speech_no_index["log_months_to_election"].mean()
    topic_ref = df_speech_no_index["topic_label"].mode()[0]
    print(topic_ref)

    parties = df_speech_no_index["party"].dropna().unique()

    # Prediction data for model 1: without gov_status
    pred_df = pd.DataFrame({
        "party": parties,
        "election_term": election_term_ref,
        "log_months_to_election": log_months_ref,
        "topic_label": topic_ref
    })

    # Prediction data for model 2: with gov_status
    pred_df_2 = pd.DataFrame({
        "party": parties,
        "gov_status": gov_ref,
        "election_term": election_term_ref,
        "log_months_to_election": log_months_ref,
        "topic_label": topic_ref
    })

    # Model 1: without government status
    model = smf.ols(
        """
        moral_rating ~ C(party, Treatment(reference='Socialdemokraterna'))
        + C(election_term)
        + C(topic_label)
        + log_months_to_election
        """,
        data=df_speech_no_index
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": speech_groups}
    )

    # Model 2: with government status
    model2 = smf.ols(
        """
        moral_rating ~ C(party, Treatment(reference='Socialdemokraterna'))
        + C(election_term)
        + C(topic_label)
        + gov_status
        + log_months_to_election
        """,
        data=df_speech_no_index
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": speech_groups}
    )

    # Predictions for model 1
    pred_summary = model.get_prediction(pred_df).summary_frame(alpha=0.05)

    pred_df["pred"] = pred_summary["mean"].values
    pred_df["ci_low"] = pred_summary["mean_ci_lower"].values
    pred_df["ci_high"] = pred_summary["mean_ci_upper"].values
    pred_df["model"] = "Without government status"

    # Predictions for model 2
    pred_summary_2 = model2.get_prediction(pred_df_2).summary_frame(alpha=0.05)

    pred_df_2["pred"] = pred_summary_2["mean"].values
    pred_df_2["ci_low"] = pred_summary_2["mean_ci_lower"].values
    pred_df_2["ci_high"] = pred_summary_2["mean_ci_upper"].values
    pred_df_2["model"] = "With government status"

    # Combine
    pred_all = pd.concat([pred_df, pred_df_2], ignore_index=True)

    # English party names
    pred_all["party_en"] = pred_all["party"].map(PARTY_EN).fillna(pred_all["party"])

    # Sort parties by model 1 predictions
    party_order = (
        pred_all[pred_all["model"] == "With government status"]
        .sort_values("pred")["party_en"]
        .tolist()
    )

    pred_all["party_en"] = pd.Categorical(
        pred_all["party_en"],
        categories=party_order,
        ordered=True
    )

    pred_all = pred_all.sort_values("party_en").copy()

    # Numeric y positions
    party_to_y = {party: i for i, party in enumerate(party_order)}
    pred_all["y"] = pred_all["party_en"].map(party_to_y).astype(float)

    # Swapped visual order:
    # With government status appears above / first in legend
    model_order = [
        "Without government status",
        "With government status"
    ]

    offsets = {
        "Without government status": -0.12,
        "With government status": 0.12
    }

    colors = {
        "With government status": "#DD8452",
        "Without government status": "#4C72B0"
    }

    pred_all["y_offset"] = pred_all["y"] + pred_all["model"].map(offsets)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 7))

    for model_name in model_order:
        data = pred_all[pred_all["model"] == model_name]

        ax.hlines(
            y=data["y_offset"],
            xmin=data["ci_low"],
            xmax=data["ci_high"],
            color=colors[model_name],
            alpha=0.85,
            linewidth=2,
            label=None
        )

        ax.scatter(
            data["pred"],
            data["y_offset"],
            color=colors[model_name],
            s=45,
            label=model_name,
            zorder=3
        )

    # Axis formatting
    ax.set_yticks(range(len(party_order)))
    ax.set_yticklabels(party_order)

    ax.set_xlabel("Predicted moral rating")
    ax.set_ylabel("")
    ax.legend(frameon=False, loc="best")

    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "predicted_speech_level_two_models.jpg",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# Event study

In [ ]:
if TOPICS_AVAILABLE:
    # Put periods in chronological order
    periods = sorted(
        [
            {
                "start": start,
                "end": end,
                "government_parties": set(parties)
            }
            for (start, end), parties in government_periods.items()
        ],
        key=lambda x: x["start"]
    )

    # All parties appearing in the government-period dictionary
    all_parties = sorted(
        set().union(*(p["government_parties"] for p in periods))
    )

    transitions = []

    for previous, current in zip(periods[:-1], periods[1:]):

        transition_date = current["start"]

        previous_gov = previous["government_parties"]
        current_gov = current["government_parties"]

        for party in all_parties:

            was_government = party in previous_gov
            is_government = party in current_gov

            # Only retain actual status changes
            if was_government != is_government:

                direction = (
                    "gov_to_opp"
                    if was_government and not is_government
                    else "opp_to_gov"
                )

                transitions.append({
                    "party": party,
                    "transition_date": transition_date,
                    "from_status": (
                        "government" if was_government else "opposition"
                    ),
                    "to_status": (
                        "government" if is_government else "opposition"
                    ),
                    "direction": direction
                })

    transitions_df = pd.DataFrame(transitions)

    transitions_df.head()

In [ ]:
if TOPICS_AVAILABLE:
    df_event = df_speech_no_index.merge(
        transitions_df,
        on="party",
        how="inner"
    )

    df_event["event_days"] = (
        df_event["date"] - df_event["transition_date"]
    ).dt.days

    df_event["event_month"] = df_event["event_days"] / 30.44

    df_event["month_of_year"] = pd.to_datetime(
        df_event["date"]
    ).dt.month

In [ ]:
if TOPICS_AVAILABLE:
    df_event = df_event[
        df_event["event_month"].between(-24, 24)
    ].copy()

In [ ]:
if TOPICS_AVAILABLE:
    df_govopp = df_event[
        df_event["direction"] == "gov_to_opp"
    ].copy()

In [ ]:
if TOPICS_AVAILABLE:
    #create bins

    # df_govopp["event_bin"] = pd.cut(
    #     df_govopp["event_month"],
    #     bins=[-12, -9, -6, -3, 0, 3, 6, 9, 12],
    #     labels=[
    #         "12_to_9_before",
    #         "9_to_6_before",
    #         "6_to_3_before",
    #         "3_to_0_before",
    #         "0_to_3_after",
    #         "3_to_6_after",
    #         "6_to_9_after",
    #         "9_to_12_after"
    #     ],
    #     right=False,
    #     include_lowest=True
    # )

    df_govopp["event_bin"] = pd.cut(
        df_govopp["event_month"],
        bins=[-24, -18, -12, -6, 0, 6, 12, 18, 24],
        labels=[
            "24_to_18_before",
            "18_to_12_before",
            "12_to_6_before",
            "6_to_0_before",
            "0_to_6_after",
            "6_to_12_after",
            "12_to_18_after",
            "18_to_24_after"
        ],
        right=False,
        include_lowest=True
    )

In [ ]:
if TOPICS_AVAILABLE:
    # set index again
    df_govopp = df_govopp.set_index(["person_id", "date"])

In [ ]:
if TOPICS_AVAILABLE:
    event_dummies = pd.get_dummies(
        df_govopp["event_bin"],
        prefix="event",
        dtype=float
    )

    #event_dummies = event_dummies.drop(
    #    columns="event_3_to_0_before"
    #)

    df_govopp = pd.concat(
        [df_govopp, event_dummies],
        axis=1
    )

In [ ]:
if TOPICS_AVAILABLE:
    mod_event = PanelOLS.from_formula(
        """
        moral_rating ~ 1
        + C(event_bin, Treatment(reference='6_to_0_before'))
        + EntityEffects
        + C(topic_label)
        """,
        data=df_govopp
    ).fit(
        cov_type="clustered",
        cluster_entity=True,
        cluster_time=True
    )
    mod_event.summary

In [ ]:
if TOPICS_AVAILABLE:
    # Now switch direction

    df_oppgov = df_event[
        df_event["direction"] == "opp_to_gov"
    ].copy()
    df_oppgov.head()

In [ ]:
if TOPICS_AVAILABLE:
    #create bins

    df_oppgov["event_bin"] = pd.cut(
        df_oppgov["event_month"],
        bins=[-24, -18, -12, -6, 0, 6, 12, 18, 24],
        labels=[
            "24_to_18_before",
            "18_to_12_before",
            "12_to_6_before",
            "6_to_0_before",
            "0_to_6_after",
            "6_to_12_after",
            "12_to_18_after",
            "18_to_24_after"
        ],
        right=False,
        include_lowest=True
    )

In [ ]:
if TOPICS_AVAILABLE:
    # set index again
    df_oppgov = df_oppgov.set_index(["person_id", "date"])

In [ ]:
if TOPICS_AVAILABLE:
    event_dummies = pd.get_dummies(
        df_oppgov["event_bin"],
        prefix="event",
        dtype=float
    )

    #event_dummies = event_dummies.drop(
    #    columns="event_3_to_0_before"
    #)

    df_oppgov = pd.concat(
        [df_oppgov, event_dummies],
        axis=1
    )

In [ ]:
if TOPICS_AVAILABLE:
    mod_event2 = PanelOLS.from_formula(
        """
        moral_rating ~ 1
        + C(event_bin, Treatment(reference='6_to_0_before'))
        + EntityEffects
        + C(topic_label)
        """,
        data=df_oppgov
    ).fit(
        cov_type="clustered",
        cluster_entity=True,
        cluster_time=True
    )
    mod_event2.summary

In [ ]:
if TOPICS_AVAILABLE:
    def stars(p):
        if p < 0.01:
            return "***"
        elif p < 0.05:
            return "**"
        elif p < 0.10:
            return "*"
        return ""


    def get_n_entities(model):
        return (
            model.model.dependent.dataframe
            .index
            .get_level_values(0)
            .nunique()
        )


    def event_table(mod1, mod2):

        models = {
            "Gov $\\rightarrow$ Opp": mod1,
            "Opp $\\rightarrow$ Gov": mod2
        }

        event_labels = {
            "24_to_18_before": "24--18 months before",
            "18_to_12_before": "18--12 months before",
            "12_to_6_before": "12--6 months before",
            "0_to_6_after": "0--6 months after",
            "6_to_12_after": "6--12 months after",
            "12_to_18_after": "12--18 months after",
            "18_to_24_after": "18--24 months after"
        }

        rows = []

        for event_name, nice_label in event_labels.items():

            coef_row = {"Period": nice_label}
            se_row = {"Period": ""}

            for model_name, model in models.items():

                matches = [
                    p for p in model.params.index
                    if "event_bin" in p and event_name in p
                ]

                if matches:
                    param = matches[0]

                    b = model.params[param]
                    se = model.std_errors[param]
                    pval = model.pvalues[param]

                    coef_row[model_name] = f"{b:.3f}{stars(pval)}"
                    se_row[model_name] = f"({se:.3f})"

                else:
                    coef_row[model_name] = ""
                    se_row[model_name] = ""

            rows.append(coef_row)
            rows.append(se_row)

            # Insert omitted reference category after the pre-transition bins
            if event_name == "12_to_6_before":
                rows.append({
                    "Period": "6--0 months before",
                    "Gov $\\rightarrow$ Opp": "Reference",
                    "Opp $\\rightarrow$ Gov": "Reference"
                })

        # Model summary rows
        rows.extend([
            {
                "Period": "MP fixed effects",
                "Gov $\\rightarrow$ Opp": "Yes",
                "Opp $\\rightarrow$ Gov": "Yes"
            },
            {
                "Period": "Topic controls",
                "Gov $\\rightarrow$ Opp": "Yes",
                "Opp $\\rightarrow$ Gov": "Yes"
            },
            {
                "Period": "Observations",
                "Gov $\\rightarrow$ Opp": f"{int(mod1.nobs):,}",
                "Opp $\\rightarrow$ Gov": f"{int(mod2.nobs):,}"
            },
            {
                "Period": "MPs",
                "Gov $\\rightarrow$ Opp": f"{get_n_entities(mod1):,}",
                "Opp $\\rightarrow$ Gov": f"{get_n_entities(mod2):,}"
            },
            {
                "Period": r"$R^2$",
                "Gov $\\rightarrow$ Opp": f"{mod1.rsquared:.3f}",
                "Opp $\\rightarrow$ Gov": f"{mod2.rsquared:.3f}"
            }
        ])

        return pd.DataFrame(rows)

In [ ]:
if TOPICS_AVAILABLE:
    tab = event_table(mod_event, mod_event2)
    tab

In [ ]:
if TOPICS_AVAILABLE:
    latex = tab.to_latex(
        index=False,
        escape=False,
        column_format="lcc",
        caption="Event-study estimates around government--opposition transitions.",
        label="tab:event_study"
    )

    print(latex)
    (TABLE_DIR / "table_a7_1.tex").write_text(latex, encoding="utf-8")

In [ ]:
if TOPICS_AVAILABLE:
    # plot

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    def extract_event_coefs(model):
        mapping = {
            "24_to_18_before": -21,
            "18_to_12_before": -15,
            "12_to_6_before": -9,
            "0_to_6_after": 3,
            "6_to_12_after": 9,
            "12_to_18_after": 15,
            "18_to_24_after": 21
        }

        rows = []

        for event_name, event_time in mapping.items():
            matches = [
                p for p in model.params.index
                if "event_bin" in p and event_name in p
            ]

            if matches:
                param = matches[0]

                rows.append({
                    "event_time": event_time,
                    "coef": model.params[param],
                    "se": model.std_errors[param]
                })

        # Add omitted reference period
        rows.append({
            "event_time": -3,
            "coef": 0,
            "se": 0
        })

        out = pd.DataFrame(rows).sort_values("event_time")

        out["lower"] = out["coef"] - 1.96 * out["se"]
        out["upper"] = out["coef"] + 1.96 * out["se"]

        return out

In [ ]:
if TOPICS_AVAILABLE:
    gov_opp = extract_event_coefs(mod_event)
    opp_gov = extract_event_coefs(mod_event2)

In [ ]:
if TOPICS_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.errorbar(
        gov_opp["event_time"],
        gov_opp["coef"],
        yerr=1.96 * gov_opp["se"],
        fmt="o",
        capsize=3
    )

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linestyle="--", linewidth=1)

    ax.set_xlabel("Months relative to moving to opposition")
    ax.set_ylabel("Change in moralizing rhetoric")
    #ax.set_title("Government → Opposition")

    ax.set_xticks(
        [-21, -15, -9, -3, 3, 9, 15, 21]
    )
    ax.set_ylim(-0.6, 0.6)

    plt.tight_layout()
    plt.savefig(FIG_DIR / "gov-opp.jpg")
    plt.show()

In [ ]:
if TOPICS_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.errorbar(
        opp_gov["event_time"],
        opp_gov["coef"],
        yerr=1.96 * opp_gov["se"],
        fmt="o",
        capsize=3
    )

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linestyle="--", linewidth=1)

    ax.set_xlabel("Months relative to moving to government")
    ax.set_ylabel("Change in moralizing rhetoric")
    #ax.set_title("Government → Opposition")

    ax.set_xticks(
        [-21, -15, -9, -3, 3, 9, 15, 21]
    )
    ax.set_ylim(-0.6, 0.6)
    plt.tight_layout()
    out_path = FIG_DIR / "opp-gov.jpg"
    plt.savefig(out_path)
    plt.show()

In [ ]:


from stargazer.stargazer import Stargazer
import statsmodels.formula.api as smf

df_reg_ = df_reg[df_reg['election_term'] != 8]
term_dummies = pd.get_dummies(df_reg_['election_term'], dtype=int)
# -------------------------
person_codes = df_reg_["person_id"].astype("category").cat.codes
month_codes = df_reg_["year_month"].astype("category").cat.codes
groups = np.column_stack((person_codes, month_codes))


df_reg_['election_term'] = df_reg_["election_term"].astype("category").cat.codes
X = df_reg_


mod1 = smf.ols(
    "moral_rating ~  gov_status + log_months_to_election + C(election_term)",
    data=X
).fit(cov_type="cluster",
    cov_kwds={"groups":groups})

mod2 = smf.ols(
    "moral_rating ~  gov_status + log_months_to_election + C(election_term) + C(party, Treatment(reference='Socialdemokraterna'))",
    data=X
).fit(cov_type="cluster",
    cov_kwds={"groups":groups})



mod3 =smf.ols(
    "moral_rating ~  gov_status * log_months_to_election + C(election_term) + C(party, Treatment(reference='Socialdemokraterna')) ",
    data=X
).fit(cov_type="cluster",
    cov_kwds={"groups":groups})


stargazer = Stargazer([mod1, mod2, mod3])

latex_table = stargazer.render_latex()

print(latex_table)

(TABLE_DIR / "table_a6_1.tex").write_text(latex_table, encoding="utf-8")
pd.DataFrame({"estimate": mod1.params, "se": mod1.bse}).to_csv(TABLE_DIR / "mp_month_coefficients.csv")


In [ ]:
df_plot = df.copy()
print(df_plot)
df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")
df_plot = df_plot.dropna(subset=["date", "moral_rating", "anti_establishment"])


def nearest_election(date, elections):
    deltas = elections - date          # TimedeltaIndex
    idx = np.argmin(np.abs(deltas))    # numpy handles abs()
    return elections[idx]

df_plot["election_date"] = df_plot["date"].apply(
    lambda d: nearest_election(d, election_dates)
)

In [ ]:
df_plot["event_month"] = (
    (df_plot["date"].dt.year - df_plot["election_date"].dt.year) * 12 +
    (df_plot["date"].dt.month - df_plot["election_date"].dt.month)
)
WINDOW = 88  # ±24 months
df_plot = df_plot[df_plot["event_month"].between(-WINDOW, WINDOW)]

In [ ]:
df_plot["event_week"] = (df_plot["date"] - df_plot["election_date"]).dt.days // 7

WINDOW =  44# number of weeks before/after election
df_plot = df_plot[df_plot["event_week"].between(-WINDOW, WINDOW-20)]

In [ ]:
# aggregate across all the elections

# df_plot["anti_establishment"] = df_plot["anti_establishment"].map({
#     1: "Anti-establishment parties",
#     0: "Mainstream parties"
# })

summary = (
    df_plot
    .groupby(["event_week"])["moral_rating"]
    .mean()
    .reset_index()
    .sort_values("event_week")
)


In [ ]:
data = summary.sort_values("event_week")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

fig, ax = plt.subplots(figsize=(12, 5))

# Grey election window
ax.axvspan(-12, 4, color="0.9", alpha=0.8, zorder=0)

window_size = 3  # rolling window in weeks


data = data.sort_values("event_week").copy()

# Compute rolling mean
data["moral_rating_smooth"] = data["moral_rating"].rolling(
    window=window_size, center=True, min_periods=1
).mean()

# Compute rolling standard error
data["se"] = data["moral_rating"].rolling(
    window=window_size, center=True, min_periods=1
).sem()  # pandas' sem = std / sqrt(n)

pre = data[data["event_week"] < -11]
post = data[data["event_week"] >= 4]

color = None

# solid pre with shaded SE
if not pre.empty:
    line, = ax.plot(
        pre["event_week"],
        pre["moral_rating_smooth"],
        lw=2,
       # label=gov_label
    )
    color = line.get_color()
    ax.fill_between(
        pre["event_week"],
        pre["moral_rating_smooth"] - pre["se"],
        pre["moral_rating_smooth"] + pre["se"],
        color=color,
        alpha=0.1
    )

# solid post with shaded SE
if not post.empty:
    ax.plot(
        post["event_week"],
        post["moral_rating_smooth"],
        lw=2,
        color=color
    )
    ax.fill_between(
        post["event_week"],
        post["moral_rating_smooth"] - post["se"],
        post["moral_rating_smooth"] + post["se"],
        color=color,
        alpha=0.1
    )

# dashed bridge
if not pre.empty and not post.empty:
    ax.plot(
        [pre["event_week"].iloc[-1], post["event_week"].iloc[0]],
        [pre["moral_rating_smooth"].iloc[-1], post["moral_rating_smooth"].iloc[0]],
        lw=2,
        linestyle="--",
        color=color
    )

# Election reference line
ax.axvline(0, linestyle="--", color="black", alpha=0.6)

plt.xticks(range(int(summary["event_week"].min()), 
                 int(summary["event_week"].max()), 10))
#ax.set_title("Average Moral Rating Around Elections (Weekly Averages, Pooled; 3-Week Rolling Mean)")
ax.set_xlabel("Weeks relative to election")
ax.set_ylabel("Average moral rating")
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
#ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "pooled.jpg", dpi=300)
plt.show()

data.to_csv(TABLE_DIR / "election_week_means.csv", index=False)
